# THUMOS14 Two-Stage TAL — Training Notebook

**Stage 1 (BiMambaGlobalScanner)** scans long video features for action proposals.
**Stage 2 (TransformerRefiner)** refines proposal boundaries on 64-frame windows.

## Prerequisites (run in a terminal, not here)

```bash
# 1. Dataset must be flat at /workspace/THUMOS14
ls /workspace/THUMOS14/    # expect: features  gt.json  split_test.txt  split_train.txt

# 2. Stage to RAM for fast random reads (~30s, repeats per pod)
mkdir -p /dev/shm/THUMOS14
cp -r /workspace/THUMOS14/* /dev/shm/THUMOS14/

# 3. mamba-ssm + causal-conv1d installed (optional — falls back to transformers if missing)
python -c "import mamba_ssm; print('mamba_ssm OK')"
```

## Recommended workflow

VS Code's Jupyter over SSH is fragile for long cells (websocket drops → kernel
appears frozen with 0% CPU). Use this notebook for **debugging and short cells**.
For multi-hour training, copy the relevant cells to a `.py` file and run via
`tmux + python -u train.py 2>&1 | tee train.log`.


In [ ]:
import os, sys, json, time, random, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm  # plain tqdm — NOT tqdm.auto (the auto IPyWidgets backend freezes Jupyter over SSH)

# ── Paths ──────────────────────────────────────────────────────────────
DATA_ROOT_PERSISTENT = '/workspace/THUMOS14'    # MooseFS, slow random reads
DATA_ROOT_FAST       = '/dev/shm/THUMOS14'      # tmpfs, ~100x faster random reads
DATA_ROOT = DATA_ROOT_FAST if os.path.exists(DATA_ROOT_FAST) else DATA_ROOT_PERSISTENT

CACHE_DIR    = '/workspace/cache'
CACHE_TRAIN  = f'{CACHE_DIR}/cache_train.pt'
CACHE_TEST   = f'{CACHE_DIR}/cache_test.pt'

CKPT_DIR     = '/workspace/checkpoints'
SCANNER_CKPT = f'{CKPT_DIR}/mamba_scanner_best.pth'
REFINER_CKPT = f'{CKPT_DIR}/transformer_refiner_best.pth'
PRED_FILE    = '/workspace/submission_predictions.json'

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,  exist_ok=True)

# ── Data / model constants ─────────────────────────────────────────────
FPS, CLIP_LENGTH, WINDOW_SIZE = 25.0, 16, 64
INPUT_DIM = 2048   # I3D RGB(1024) + Flow(1024)

# ── Hyperparameters ────────────────────────────────────────────────────
BATCH_SIZE      = 4
MAX_SEQ_LEN     = 1024     # random crop during training; full length used at inference
EPOCHS_STAGE1   = 10
EPOCHS_STAGE2   = 15
LR              = 1e-4
WEIGHT_DECAY    = 1e-4
GRAD_CLIP       = 1.0
FOCAL_ALPHA     = 0.25
FOCAL_GAMMA     = 2.0
SCORE_THRESHOLD = 0.5
NMS_IOU_THR     = 0.4

# ── Device ─────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}, {p.total_memory/1e9:.1f} GB VRAM')
print(f'DATA_ROOT = {DATA_ROOT}')


In [ ]:
# Build per-split caches (.pt files containing list of (fused_feat, frame_mask, vid_id)).
# Idempotent: skips work if the cache file already exists.
# This isolates ALL filesystem I/O to one short cell — kernel never blocks here again.

def build_cache(split, out_path):
    if os.path.exists(out_path):
        print(f'[{split}] cache exists at {out_path} — skipping rebuild')
        return
    rgb_dir, flow_dir = f'{DATA_ROOT}/features/{split}/rgb', f'{DATA_ROOT}/features/{split}/flow'
    with open(f'{DATA_ROOT}/split_{split}.txt') as f:
        vids = [l.strip() for l in f if l.strip()]
    with open(f'{DATA_ROOT}/gt.json') as f:
        gt = json.load(f)

    cache, skipped, t0 = [], 0, time.time()
    print(f'[{split}] caching {len(vids)} videos from {DATA_ROOT}...', flush=True)
    for i, vid in enumerate(vids):
        if i % 25 == 0:
            print(f'  {i}/{len(vids)}  cached={len(cache)}  skipped={skipped}', flush=True)
        rgb_p, flow_p = f'{rgb_dir}/{vid}.npy', f'{flow_dir}/{vid}.npy'
        if not (os.path.exists(rgb_p) and os.path.exists(flow_p)):
            skipped += 1; continue
        try:
            rgb, flow = np.load(rgb_p), np.load(flow_p)
        except Exception as e:
            print(f'  [WARN] {vid}: {e}', flush=True); skipped += 1; continue
        fused = torch.from_numpy(np.concatenate([rgb, flow], axis=-1)).float()
        T = fused.shape[0]
        mask = torch.zeros(T, dtype=torch.float32)
        for ann in gt.get('database', {}).get(vid, {}).get('annotations', []):
            s = max(0, min(int(ann['segment'][0] * FPS / CLIP_LENGTH), T - 1))
            e = max(0, min(int(ann['segment'][1] * FPS / CLIP_LENGTH), T - 1))
            if s <= e:
                mask[s:e+1] = 1.0
        cache.append((fused, mask, vid))
    print(f'[{split}] done: cached={len(cache)} skipped={skipped} ({time.time()-t0:.1f}s)', flush=True)
    torch.save(cache, out_path)
    print(f'[{split}] saved to {out_path}', flush=True)

build_cache('train', CACHE_TRAIN)
build_cache('test',  CACHE_TEST)


In [ ]:
class CachedScannerDataset(Dataset):
    """Loads pre-cached features list. Random crop to MAX_SEQ_LEN if max_len given."""
    def __init__(self, cache_path, max_len=None):
        self.cache = torch.load(cache_path, weights_only=False)
        self.max_len = max_len

    def __len__(self): return len(self.cache)

    def __getitem__(self, idx):
        fused, mask, vid = self.cache[idx]
        if self.max_len is not None and fused.shape[0] > self.max_len:
            start = random.randint(0, fused.shape[0] - self.max_len)
            fused = fused[start:start + self.max_len]
            mask  = mask [start:start + self.max_len]
        return fused, mask, vid


def pad_collate(batch):
    """Pads variable-length (T, 2048) features and (T,) masks to a common Tmax.
    Returns: feats (B, Tmax, 2048), masks (B, Tmax), pad_mask (B, Tmax) bool, ids list.
    pad_mask is True at PADDED positions (use ~pad_mask for valid positions)."""
    feats, masks, ids = zip(*batch)
    lengths   = torch.tensor([f.shape[0] for f in feats], dtype=torch.long)
    feats_p   = pad_sequence(feats, batch_first=True)                       # (B, Tmax, 2048)
    masks_p   = pad_sequence(masks, batch_first=True, padding_value=0.0)    # (B, Tmax)
    pad_mask  = torch.arange(feats_p.size(1))[None, :] >= lengths[:, None]  # (B, Tmax) bool
    return feats_p, masks_p, pad_mask, list(ids)


train_dataset = CachedScannerDataset(CACHE_TRAIN, max_len=MAX_SEQ_LEN)
train_loader  = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=False, collate_fn=pad_collate,
)

# Sanity check
feats, mask, pad_mask, ids = next(iter(train_loader))
print(f'feats   = {tuple(feats.shape)}')
print(f'mask    = {tuple(mask.shape)}    positives = {int(mask.sum())}/{mask.numel()}')
print(f'pad_mask= {tuple(pad_mask.shape)} valid     = {int((~pad_mask).sum())}/{pad_mask.numel()}')
print(f'ids     = {ids}')


In [ ]:
class FocalLoss(nn.Module):
    """Sigmoid focal loss with proper class-aware alpha_t (Lin et al. 2017).
    alpha=0.25 by convention (focusing term already up-weights hard positives)."""
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha, self.gamma, self.reduction = alpha, gamma, reduction

    def forward(self, logits, targets):
        bce  = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p    = torch.sigmoid(logits)
        p_t  = p * targets + (1 - p) * (1 - targets)
        a_t  = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = a_t * (1 - p_t).pow(self.gamma) * bce
        if self.reduction == 'mean': return loss.mean()
        if self.reduction == 'sum':  return loss.sum()
        return loss


In [ ]:
# Use the real mamba-ssm CUDA kernels if available, otherwise fall back to
# transformers.MambaModel (slower but correct). The fallback explicitly sets
# num_hidden_layers — without it, HF defaults to 32 layers per direction
# (= 64 total), which was the silent VRAM-killer in the original notebook.

try:
    from mamba_ssm import Mamba as MambaSSMBlock
    HAS_MAMBA_SSM = True
    print('[Mamba] using mamba_ssm (CUDA selective scan)')
except Exception as e:
    HAS_MAMBA_SSM = False
    from transformers import MambaConfig, MambaModel
    print(f'[Mamba] mamba_ssm unavailable ({e}); using transformers fallback')


class BiMambaGlobalScanner(nn.Module):
    def __init__(self, input_dim=2048, d_model=256, n_layers=4,
                 d_state=16, d_conv=4, expand=2, kernel_size=7):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.temporal_smooth = nn.Conv1d(
            d_model, d_model, kernel_size=kernel_size,
            padding=kernel_size // 2, groups=d_model,
        )

        if HAS_MAMBA_SSM:
            self.fwd_blocks = nn.ModuleList([
                MambaSSMBlock(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
                for _ in range(n_layers)])
            self.bwd_blocks = nn.ModuleList([
                MambaSSMBlock(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
                for _ in range(n_layers)])
            self.fwd_norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
            self.bwd_norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        else:
            cfg = MambaConfig(
                hidden_size=d_model, state_size=d_state, conv_kernel=d_conv,
                expand=expand, num_hidden_layers=n_layers,
            )
            self.fwd_hf = MambaModel(cfg)
            self.bwd_hf = MambaModel(cfg)

        self.classifier = nn.Linear(d_model * 2, 1)

    def forward(self, x):                              # x: (B, T, input_dim)
        x = self.input_proj(x)                         # (B, T, d_model)
        x = self.temporal_smooth(x.transpose(1, 2)).transpose(1, 2)

        if HAS_MAMBA_SSM:
            f = x
            for blk, ln in zip(self.fwd_blocks, self.fwd_norms):
                f = f + blk(ln(f))                     # pre-norm + residual
            b = torch.flip(x, dims=[1])
            for blk, ln in zip(self.bwd_blocks, self.bwd_norms):
                b = b + blk(ln(b))
            b = torch.flip(b, dims=[1])
        else:
            f = self.fwd_hf(inputs_embeds=x).last_hidden_state
            b_in = torch.flip(x, dims=[1])
            b = torch.flip(self.bwd_hf(inputs_embeds=b_in).last_hidden_state, dims=[1])

        h = torch.cat([f, b], dim=-1)                  # (B, T, 2*d_model)
        return self.classifier(h).squeeze(-1)          # (B, T)


# Quick build & forward sanity check
_m = BiMambaGlobalScanner().to(DEVICE)
print(f'Stage1 params: {sum(p.numel() for p in _m.parameters())/1e6:.2f}M')
with torch.no_grad():
    _x = torch.randn(2, 128, 2048, device=DEVICE)
    print(f'forward OK: in={tuple(_x.shape)} out={tuple(_m(_x).shape)}')
del _m


In [ ]:
mamba_model = BiMambaGlobalScanner().to(DEVICE)
focal_none  = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, reduction='none').to(DEVICE)
optimizer   = optim.AdamW(mamba_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler      = torch.amp.GradScaler('cuda')

best_loss = float('inf')

for epoch in range(EPOCHS_STAGE1):
    mamba_model.train()
    total, n = 0.0, 0
    pbar = tqdm(train_loader, desc=f'Stage1 {epoch+1}/{EPOCHS_STAGE1}',
                mininterval=1.0, maxinterval=10.0)
    for feats, target_mask, pad_mask, _ in pbar:
        feats       = feats.to(DEVICE, non_blocking=True)
        target_mask = target_mask.to(DEVICE, non_blocking=True)
        pad_mask    = pad_mask.to(DEVICE, non_blocking=True)
        valid       = (~pad_mask).float()              # 1 where real, 0 where padded

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            logits    = mamba_model(feats)             # (B, T)
            per_frame = focal_none(logits.float(), target_mask)
            loss      = (per_frame * valid).sum() / valid.sum().clamp(min=1.0)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(mamba_model.parameters(), GRAD_CLIP)
        scaler.step(optimizer); scaler.update()

        total += loss.item(); n += 1
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    avg = total / max(1, n)
    print(f'epoch {epoch+1}: avg_loss={avg:.4f}')
    if avg < best_loss:
        best_loss = avg
        torch.save(mamba_model.state_dict(), SCANNER_CKPT)
        print(f'  saved best to {SCANNER_CKPT}')

print('Stage 1 training complete.')


In [ ]:
class RefinerDataset(Dataset):
    """Builds (window_features, normalized_target_offsets) pairs from the
    cached training set. The window is centered on each GT segment with a small
    random jitter to simulate Stage-1 proposal noise."""

    def __init__(self, cache_path, window_size=64, jitter=8):
        self.cache       = torch.load(cache_path, weights_only=False)
        self.window_size = window_size
        self.jitter      = jitter
        self.snippets    = []  # list of (cache_idx, gt_start_idx, gt_end_idx)

        with open(f'{DATA_ROOT}/gt.json') as f:
            gt = json.load(f)

        for ci, (fused, _, vid) in enumerate(self.cache):
            T = fused.shape[0]
            for ann in gt.get('database', {}).get(vid, {}).get('annotations', []):
                s = max(0, min(int(ann['segment'][0] * FPS / CLIP_LENGTH), T - 1))
                e = max(0, min(int(ann['segment'][1] * FPS / CLIP_LENGTH), T - 1))
                if s < e:
                    self.snippets.append((ci, s, e))
        print(f'Refiner snippets built: {len(self.snippets)}')

    def __len__(self): return len(self.snippets)

    def __getitem__(self, idx):
        ci, s, e = self.snippets[idx]
        fused, _, _ = self.cache[ci]
        T = fused.shape[0]
        center = (s + e) // 2 + random.randint(-self.jitter, self.jitter)
        w_start = max(0, center - self.window_size // 2)
        w_end   = w_start + self.window_size
        if w_end > T:
            w_end   = T
            w_start = max(0, w_end - self.window_size)
        snip = fused[w_start:w_end]
        if snip.shape[0] < self.window_size:
            pad = torch.zeros(self.window_size - snip.shape[0], snip.shape[1])
            snip = torch.cat([snip, pad], dim=0)
        # Targets normalized to [0, 1] relative to window
        s_t = max(0.0, min(1.0, (s - w_start) / self.window_size))
        e_t = max(0.0, min(1.0, (e - w_start) / self.window_size))
        return snip, torch.tensor([s_t, e_t], dtype=torch.float32)


refiner_dataset = RefinerDataset(CACHE_TRAIN, window_size=WINDOW_SIZE, jitter=8)
refiner_loader  = DataLoader(refiner_dataset, batch_size=32, shuffle=True,
                             num_workers=0, pin_memory=False)
print(f'Refiner snippets: {len(refiner_dataset)}, batches/epoch: {len(refiner_loader)}')


In [ ]:
class TransformerRefiner(nn.Module):
    def __init__(self, input_dim=2048, d_model=256, nhead=4, num_layers=2,
                 window_size=64, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.local_conv = nn.Conv1d(d_model, d_model, kernel_size=3, padding=1)
        self.pos_embed  = nn.Parameter(torch.randn(1, window_size, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        # Sigmoid keeps output in [0, 1] of the window. If you want to allow
        # extending beyond the window, swap to (Linear -> 2 raw outputs) and
        # interpret as (center, log_width).
        self.regressor = nn.Sequential(
            nn.Linear(d_model, 64), nn.GELU(),
            nn.Linear(64, 2), nn.Sigmoid(),
        )

    def forward(self, x):                              # x: (B, W, 2048)
        x = self.input_proj(x)
        x = self.local_conv(x.transpose(1, 2)).transpose(1, 2)
        x = x + self.pos_embed[:, :x.size(1)]
        x = self.encoder(x)
        x = x.mean(dim=1)                              # global pool over window
        return self.regressor(x)                       # (B, 2)


In [ ]:
class GIoULoss1D(nn.Module):
    """Generalized IoU loss for 1D intervals. Canonicalises predicted (start, end)
    via min/max so the loss remains valid even before the model learns ordering."""
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, preds, targets):
        ps, pe = preds[:, 0], preds[:, 1]
        ts, te = targets[:, 0], targets[:, 1]
        s_lo = torch.minimum(ps, pe)
        e_hi = torch.maximum(ps, pe)
        inter = (torch.minimum(e_hi, te) - torch.maximum(s_lo, ts)).clamp(min=0)
        union = (e_hi - s_lo) + (te - ts) - inter + self.eps
        iou   = inter / union
        enc   = torch.maximum(e_hi, te) - torch.minimum(s_lo, ts) + self.eps
        giou  = iou - (enc - union) / enc
        return (1.0 - giou).mean()


In [ ]:
refiner = TransformerRefiner(window_size=WINDOW_SIZE).to(DEVICE)
giou    = GIoULoss1D().to(DEVICE)
opt2    = optim.AdamW(refiner.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler2 = torch.amp.GradScaler('cuda')

print(f'Refiner params: {sum(p.numel() for p in refiner.parameters())/1e6:.2f}M')

best_giou = float('inf')

for epoch in range(EPOCHS_STAGE2):
    refiner.train()
    total, mae_sum, n = 0.0, 0.0, 0
    pbar = tqdm(refiner_loader, desc=f'Stage2 {epoch+1}/{EPOCHS_STAGE2}',
                mininterval=1.0, maxinterval=10.0)
    for snip, tgt in pbar:
        snip = snip.to(DEVICE, non_blocking=True)
        tgt  = tgt.to(DEVICE, non_blocking=True)

        opt2.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            pred = refiner(snip)
            loss = giou(pred.float(), tgt)

        scaler2.scale(loss).backward()
        scaler2.unscale_(opt2)
        torch.nn.utils.clip_grad_norm_(refiner.parameters(), GRAD_CLIP)
        scaler2.step(opt2); scaler2.update()

        with torch.no_grad():
            mae_sum += (pred.float() - tgt).abs().mean().item()
        total += loss.item(); n += 1
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    avg, avg_mae = total / max(1, n), mae_sum / max(1, n)
    print(f'epoch {epoch+1}: giou_loss={avg:.4f}  mae={avg_mae:.4f}')
    if avg < best_giou:
        best_giou = avg
        torch.save(refiner.state_dict(), REFINER_CKPT)
        print(f'  saved best to {REFINER_CKPT}')

print('Stage 2 training complete.')


In [ ]:
def extract_proposals(probs, threshold=SCORE_THRESHOLD):
    """Connected-component proposal extraction. Returns list of (start, end)
    indices using exclusive-end convention (Python slicing)."""
    mask = (probs > threshold).astype(np.int8)
    if mask.sum() == 0:
        return []
    padded = np.concatenate([[0], mask, [0]])
    diffs  = np.diff(padded.astype(np.int16))
    starts = np.where(diffs ==  1)[0].tolist()
    ends   = np.where(diffs == -1)[0].tolist()   # exclusive end
    return list(zip(starts, ends))


def nms_1d(segments, scores, iou_thr=NMS_IOU_THR):
    if not segments: return []
    segs   = np.asarray(segments, dtype=np.float32)
    scores = np.asarray(scores,   dtype=np.float32)
    order  = np.argsort(-scores)
    keep   = []
    while order.size > 0:
        i = order[0]; keep.append(int(i))
        if order.size == 1: break
        rest = order[1:]
        s1, e1 = segs[i]
        s2, e2 = segs[rest, 0], segs[rest, 1]
        inter  = np.clip(np.minimum(e1, e2) - np.maximum(s1, s2), 0, None)
        union  = (e1 - s1) + (e2 - s2) - inter
        iou    = np.where(union > 0, inter / np.maximum(union, 1e-9), 0.0)
        order  = rest[iou < iou_thr]
    return keep


# Re-construct & load best checkpoints
mamba_model = BiMambaGlobalScanner().to(DEVICE)
mamba_model.load_state_dict(torch.load(SCANNER_CKPT, map_location=DEVICE, weights_only=True))
mamba_model.eval()

refiner = TransformerRefiner(window_size=WINDOW_SIZE).to(DEVICE)
refiner.load_state_dict(torch.load(REFINER_CKPT, map_location=DEVICE, weights_only=True))
refiner.eval()

test_cache  = torch.load(CACHE_TEST, weights_only=False)
predictions = {}

with torch.no_grad():
    for fused, _, vid in tqdm(test_cache, desc='Inference', mininterval=1.0):
        T = fused.shape[0]
        feats = fused.unsqueeze(0).to(DEVICE)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            logits = mamba_model(feats)
        probs = torch.sigmoid(logits.float()).squeeze(0).cpu().numpy()

        proposals = extract_proposals(probs, SCORE_THRESHOLD)
        if not proposals:
            predictions[vid] = []; continue

        refined = []
        for ps, pe in proposals:
            center = (ps + pe) // 2
            w_start = max(0, center - WINDOW_SIZE // 2)
            w_end   = w_start + WINDOW_SIZE
            if w_end > T:
                w_end   = T
                w_start = max(0, w_end - WINDOW_SIZE)
            snip = fused[w_start:w_end]
            if snip.shape[0] < WINDOW_SIZE:
                pad = torch.zeros(WINDOW_SIZE - snip.shape[0], snip.shape[1])
                snip = torch.cat([snip, pad], dim=0)
            snip = snip.unsqueeze(0).to(DEVICE)
            with torch.amp.autocast('cuda', dtype=torch.float16):
                offsets = refiner(snip).float().squeeze(0).cpu().numpy()

            r_s = w_start + offsets[0] * WINDOW_SIZE
            r_e = w_start + offsets[1] * WINDOW_SIZE
            seg_start = float(r_s * CLIP_LENGTH / FPS)
            seg_end   = float(r_e * CLIP_LENGTH / FPS)
            if seg_end <= seg_start: continue
            score = float(probs[ps:pe].mean())
            refined.append({'segment': [seg_start, seg_end], 'score': score})

        if refined:
            kept = nms_1d([r['segment'] for r in refined],
                          [r['score']   for r in refined])
            refined = [refined[k] for k in kept]
        predictions[vid] = refined

with open(PRED_FILE, 'w') as f:
    json.dump(predictions, f)
n_segs = sum(len(v) for v in predictions.values())
print(f'Saved {n_segs} segments across {len(predictions)} videos -> {PRED_FILE}')


In [ ]:
GT_FILE = f'{DATA_ROOT}/gt.json'
IOU_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7]


def iou_1d(a, b):
    inter = max(0.0, min(a[1], b[1]) - max(a[0], b[0]))
    union = (a[1] - a[0]) + (b[1] - b[0]) - inter
    return inter / union if union > 0 else 0.0


with open(GT_FILE) as f:
    gt_data = json.load(f)
with open(PRED_FILE) as f:
    preds = json.load(f)

# NOTE: this is class-agnostic localization F1, not the official THUMOS14
# per-class mAP. To get per-class mAP, the model needs a per-class head and
# this loop must group by class label.
print(f'{"thr":>5} {"TP":>6} {"FP":>6} {"GT":>6} {"P":>7} {"R":>7} {"F1":>7}')
print('-' * 50)
for thr in IOU_THRESHOLDS:
    tp = fp = total_gt = 0
    for vid, gt_entry in gt_data.get('database', {}).items():
        gt_segs = [a['segment'] for a in gt_entry.get('annotations', [])]
        total_gt += len(gt_segs)
        matched = [False] * len(gt_segs)
        for p in sorted(preds.get(vid, []), key=lambda x: -x['score']):
            best_i, best_iou = -1, 0.0
            for i, g in enumerate(gt_segs):
                if matched[i]: continue
                v = iou_1d(p['segment'], g)
                if v > best_iou: best_i, best_iou = i, v
            if best_iou >= thr and best_i >= 0:
                matched[best_i] = True; tp += 1
            else:
                fp += 1
    P = tp / max(1, tp + fp)
    R = tp / max(1, total_gt)
    F1 = 2*P*R / max(1e-9, P + R)
    print(f'{thr:>5} {tp:>6} {fp:>6} {total_gt:>6} {P:>7.3f} {R:>7.3f} {F1:>7.3f}')
